### Import the libraries needed for the analysis.


In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np


### Load the customer data
Read the dataset we will use for the EDA.


In [2]:
dataset=pd.read_csv("Netflix_calculated_fields.csv")

### Check for missing data
Look for missing values before starting the calculations.


In [3]:
dataset.isna().sum()

customer_id               0
age                       0
gender                    0
subscription_type         0
watch_hours               0
last_login_days           0
region                    0
device                    0
monthly_fee               0
churned                   0
payment_method            0
number_of_profiles        0
avg_watch_time_per_day    0
favorite_genre            0
churn_status              0
revenue_lost              0
annual_revenue_lost       0
account_type              0
profile_group             0
payment_type              0
inactivity_group          0
inactive_over_30_days     0
age_group                 0
fee_per_profile           0
avg_watch_recalculated    0
viewing_level             0
high_risk_flag            0
customer_segment          0
watch_time_mismatch       0
dtype: int64

### Check for completely empty rows
See whether any rows contain no customer information.


In [4]:
dataset[dataset.isna().all(axis=1)]

,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,...,payment_type,inactivity_group,inactive_over_30_days,age_group,fee_per_profile,avg_watch_recalculated,viewing_level,high_risk_flag,customer_segment,watch_time_mismatch


### Remove empty rows
Drop rows that are completely blank so they do not affect the counts.


In [5]:
dataset = dataset.dropna(how="all")

### Check the data again
Make sure the empty rows have been handled.


In [6]:
dataset.isna().sum()

customer_id               0
age                       0
gender                    0
subscription_type         0
watch_hours               0
last_login_days           0
region                    0
device                    0
monthly_fee               0
churned                   0
payment_method            0
number_of_profiles        0
avg_watch_time_per_day    0
favorite_genre            0
churn_status              0
revenue_lost              0
annual_revenue_lost       0
account_type              0
profile_group             0
payment_type              0
inactivity_group          0
inactive_over_30_days     0
age_group                 0
fee_per_profile           0
avg_watch_recalculated    0
viewing_level             0
high_risk_flag            0
customer_segment          0
watch_time_mismatch       0
dtype: int64

# What are the most and least popular plans, genres,gender and devices?

### Compare the most and least common categories
Summarise which plans, genres, devices and genders appear most and least often.


In [7]:
popularity_summary = pd.DataFrame(index=["subscription_type", "favorite_genre", "device", "gender"],
                                  columns=["Most_popular", "Most_count", "Least_popular", "Least_count"])

plan = dataset["subscription_type"].value_counts()
popularity_summary.loc["subscription_type"] = [plan.index[0], plan.values[0], plan.index[-1], plan.values[-1]]

genre = dataset["favorite_genre"].value_counts()
popularity_summary.loc["favorite_genre"] = [genre.index[0], genre.values[0], genre.index[-1], genre.values[-1]]

device = dataset["device"].value_counts()
popularity_summary.loc["device"] = [device.index[0], device.values[0], device.index[-1], device.values[-1]]

gender = dataset["gender"].value_counts()
popularity_summary.loc["gender"] = [gender.index[0], gender.values[0], gender.index[-1], gender.values[-1]]



In [8]:
popularity_summary

,Most_popular,Most_count,Least_popular,Least_count
subscription_type,Premium,1697,Standard,1643
favorite_genre,Drama,905,Comedy,656
device,Tablet,1208,Desktop,909
gender,Female,1711,Other,1635


# What proportion of customers stayed or left?

### Calculate the overall churn split
Find how much of the customer base stayed and how much left.


In [9]:
churn_counts = dataset["churn_status"].value_counts()

prop_left = (churn_counts["Left"] / len(dataset)) * 100
prop_stayed = (churn_counts["Stayed"] / len(dataset)) * 100

print("The ratio of churn_status who left is:", float(prop_left))
print("The ratio of churn_status who Stayed is:", float(prop_stayed))


The ratio of churn_status who left is: 50.3
The ratio of churn_status who Stayed is: 49.7


# Who are the "silent subscribers"?

In [10]:
dataset.columns

Index(['customer_id', 'age', 'gender', 'subscription_type', 'watch_hours',
       'last_login_days', 'region', 'device', 'monthly_fee', 'churned',
       'payment_method', 'number_of_profiles', 'avg_watch_time_per_day',
       'favorite_genre', 'churn_status', 'revenue_lost', 'annual_revenue_lost',
       'account_type', 'profile_group', 'payment_type', 'inactivity_group',
       'inactive_over_30_days', 'age_group', 'fee_per_profile',
       'avg_watch_recalculated', 'viewing_level', 'high_risk_flag',
       'customer_segment', 'watch_time_mismatch'],
      dtype='object')

### Q1 — Find silent subscribers who still pay
Focus on retained customers with the lowest watch hours, then total their monthly fees.


In [45]:
# Question 1: Identify silent subscribers among retained customers
# Objective: Find the bottom 25% of retained subscribers by watch hours
# and calculate their total monthly subscription revenue.

# Step 1: Select retained customers (churned = 0)
active = dataset.loc[dataset["churned"] == 0]

# Step 2: Calculate the 25th percentile of watch hours among retained customers
Q1 = np.percentile(active["watch_hours"], 25)

# Step 3: Identify silent subscribers whose watch hours are at or below Q1
silent = active.loc[active["watch_hours"] <= Q1]

# Step 4: Count the number of silent subscribers
Silent_count = len(silent)

# Step 5: Calculate the total monthly subscription revenue from silent subscribers
revenue = silent["monthly_fee"].sum()

# Display the results
print("25th percentile watch-hour threshold:", Q1)
print("Number of silent subscribers:", Silent_count)
print("Total monthly revenue from silent subscribers:", round(revenue, 2))

25th percentile watch-hour threshold: 7.79
Number of silent subscribers: 623
Total monthly revenue from silent subscribers: 9209.77


# 2. How large is the inactive-but-still-paying customer base?

In [46]:
active

,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,...,payment_type,inactivity_group,inactive_over_30_days,age_group,fee_per_profile,avg_watch_recalculated,viewing_level,high_risk_flag,customer_segment,watch_time_mismatch
2,4d71f6ce-fca9-4ff7-8afa-197ac24de14b,27,Female,Standard,16.32000,10,Asia,TV,13.99,0,...,Manual,8-14 days,No,25–34,7.00,89.018182,High,No,Typical,No
5,d8079475-5be7-47e9-8782-ceb7ff61395e,58,Female,Standard,13.80000,26,Oceania,Mobile,13.99,0,...,Automatic,15-30 days,No,55–64,4.66,30.666667,Medium,No,Typical,No
6,8e63450a-13d6-4e83-bbb5-6aebde9152cb,48,Other,Basic,13.83000,20,Asia,TV,8.99,0,...,Manual,15-30 days,No,45–54,1.80,39.514286,Medium,No,Typical,No
8,0bcaad0c-545c-4ee1-85a6-75e165f39361,45,Other,Basic,9.98000,10,Asia,Mobile,8.99,0,...,Automatic,8-14 days,No,45–54,3.00,54.436364,Medium,No,Typical,Yes
10,45a03ce3-26d0-4cd5-a9bd-22d42490b612,26,Male,Premium,15.42000,36,Asia,Laptop,17.99,0,...,Automatic,31-45 days,Yes,25–34,4.50,25.005405,High,No,Typical,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4991,ac6eb271-0239-4957-b828-c89508fcfbea,62,Male,Standard,7.31000,26,South America,Tablet,13.99,0,...,Automatic,15-30 days,No,55–64,7.00,16.244444,Medium,No,Typical,No
4994,75532a22-9cd9-492c-8865-fa4ac7da2ec8,58,Other,Basic,10.90000,22,Europe,TV,8.99,0,...,Automatic,15-30 days,No,55–64,3.00,28.434783,Medium,No,Typical,No
4995,44f3ba44-b95d-4e50-a786-bac4d06f4a43,19,Female,Basic,32.77625,11,Europe,Desktop,8.99,0,...,Automatic,8-14 days,No,18–24,2.25,163.881250,High,No,Typical,Yes
4996,18779bcb-ba2b-41da-b751-e70b812061ec,67,Female,Basic,9.24000,2,North America,Desktop,8.99,0,...,Automatic,0-7 days,No,65-70,3.00,184.800000,Medium,No,Typical,Yes


### Q2 — Find inactive customers who still pay
Count retained customers who have not logged in for over 30 days and total their monthly fees.


In [47]:

# QUESTION 2: Identify retained subscribers who have not logged in
# for more than 30 days and calculate their combined monthly fees.

# Step 1: Select customers who have not churned (churned = 0).
active = dataset.loc[dataset["churned"] == 0]

# Step 2: Filter retained customers whose last login was
# more than 30 days ago.
inactive = active.loc[active["last_login_days"] > 30]

# Step 3: Count how many customers meet this condition.
inactive_count = len(inactive)

# Step 4: Calculate the total monthly fees paid by these customers.
inactive_revenue = inactive["monthly_fee"].sum()

# Step 5: Display the results.
print("Retained subscribers inactive for over 30 days:", inactive_count)
print("Combined monthly subscription fees:", round(inactive_revenue, 2))


Retained subscribers inactive for over 30 days: 579
Combined monthly subscription fees: 8591.21


# How much revenue depends on customers actively choosing to renew?
# What proportion of customers and monthly revenue comes from manual payment methods (Gift Card and Crypto)?


### Q3 — Check revenue linked to manual payments
Compare the number of manual-payment customers and their monthly fees with the whole customer base.


In [48]:

# QUESTION 3: Analyse the proportion of customers and monthly
# subscription fees associated with manual payment methods.

# Step 1: Filter customers who use Gift Card or Crypto.
manual_customers = dataset.loc[
    dataset["payment_method"].isin(["Gift Card", "Crypto"])
]

# Step 2: Count the number of manual-payment customers.
manual_count = len(manual_customers)

# Step 3: Calculate their percentage of the total customer base.
manual_payment_share = (manual_count / len(dataset)) * 100

# Step 4: Calculate the combined monthly fees from manual-payment customers.
manual_fee = manual_customers["monthly_fee"].sum()

# Step 5: Calculate the total monthly fees across all customers.
total_fee = dataset["monthly_fee"].sum()

# Step 6: Calculate the percentage of monthly fees
# associated with manual payment methods.
manual_revenue_share = (manual_fee / total_fee) * 100

# Step 7: Display the results.
print("Manual-payment customers:", manual_count)
print("Customer share (%):", round(manual_payment_share, 2))
print("Combined monthly fees:", round(manual_fee, 2))
print("Monthly fee share (%):", round(manual_revenue_share, 2))


Manual-payment customers: 1905
Customer share (%): 38.1
Combined monthly fees: 26404.95
Monthly fee share (%): 38.58


# What is the inactivity "danger zone"?
# How does churn rate change as last_login_days increases, and is there a point where churn rises sharply?

### Q4 — Count stayed and left customers by inactivity period
We need both counts in each period before we can calculate a fair churn rate.


In [52]:
# GROUP 1: 0-7 days

stayed_group1 = dataset.loc[
    (dataset["inactivity_group"] == "0-7 days") &
    (dataset["churn_status"] == "Stayed")
]

left_group1 = dataset.loc[
    (dataset["inactivity_group"] == "0-7 days") &
    (dataset["churn_status"] == "Left")
]

stayed_count1 = len(stayed_group1)
left_count1 = len(left_group1)
total_count1 = stayed_count1 + left_count1


# GROUP 2: 8-14 days

stayed_group2 = dataset.loc[
    (dataset["inactivity_group"] == "8-14 days") &
    (dataset["churn_status"] == "Stayed")
]

left_group2 = dataset.loc[
    (dataset["inactivity_group"] == "8-14 days") &
    (dataset["churn_status"] == "Left")
]

stayed_count2 = len(stayed_group2)
left_count2 = len(left_group2)
total_count2 = stayed_count2 + left_count2


# GROUP 3: 15-30 days

stayed_group3 = dataset.loc[
    (dataset["inactivity_group"] == "15-30 days") &
    (dataset["churn_status"] == "Stayed")
]

left_group3 = dataset.loc[
    (dataset["inactivity_group"] == "15-30 days") &
    (dataset["churn_status"] == "Left")
]

stayed_count3 = len(stayed_group3)
left_count3 = len(left_group3)
total_count3 = stayed_count3 + left_count3


# GROUP 4: 31-45 days

stayed_group4 = dataset.loc[
    (dataset["inactivity_group"] == "31-45 days") &
    (dataset["churn_status"] == "Stayed")
]

left_group4 = dataset.loc[
    (dataset["inactivity_group"] == "31-45 days") &
    (dataset["churn_status"] == "Left")
]

stayed_count4 = len(stayed_group4)
left_count4 = len(left_group4)
total_count4 = stayed_count4 + left_count4


# GROUP 5: 46-60 days

stayed_group5 = dataset.loc[
    (dataset["inactivity_group"] == "46-60 days") &
    (dataset["churn_status"] == "Stayed")
]

left_group5 = dataset.loc[
    (dataset["inactivity_group"] == "46-60 days") &
    (dataset["churn_status"] == "Left")
]

stayed_count5 = len(stayed_group5)
left_count5 = len(left_group5)
total_count5 = stayed_count5 + left_count5

In [53]:
print("0-7 days:", stayed_count1, left_count1, total_count1)
print("8-14 days:", stayed_count2, left_count2, total_count2)
print("15-30 days:", stayed_count3, left_count3, total_count3)
print("31-45 days:", stayed_count4, left_count4, total_count4)
print("46-60 days:", stayed_count5, left_count5, total_count5)

0-7 days: 535 83 618
8-14 days: 386 142 528
15-30 days: 985 516 1501
31-45 days: 294 890 1184
46-60 days: 285 884 1169


### Calculate churn rates for each inactivity period
Compare the percentages to see when churn rises sharply.


In [54]:
# Calculate churn rates

churn_rate_group1 = (left_count1 / total_count1) * 100
churn_rate_group2 = (left_count2 / total_count2) * 100
churn_rate_group3 = (left_count3 / total_count3) * 100
churn_rate_group4 = (left_count4 / total_count4) * 100
churn_rate_group5 = (left_count5 / total_count5) * 100


# Print churn rates

print("Group 1 (0-7 days) Churn rate in percentage is:", round(churn_rate_group1, 2))
print("Group 2 (8-14 days) Churn rate in percentage is:", round(churn_rate_group2, 2))
print("Group 3 (15-30 days) Churn rate in percentage is:", round(churn_rate_group3, 2))
print("Group 4 (31-45 days) Churn rate in percentage is:", round(churn_rate_group4, 2))
print("Group 5 (46-60 days) Churn rate in percentage is:", round(churn_rate_group5, 2))

Group 1 (0-7 days) Churn rate in percentage is: 13.43
Group 2 (8-14 days) Churn rate in percentage is: 26.89
Group 3 (15-30 days) Churn rate in percentage is: 34.38
Group 4 (31-45 days) Churn rate in percentage is: 75.17
Group 5 (46-60 days) Churn rate in percentage is: 75.62


# Engagement and churn
# Do churned customers have significantly lower daily watch time than retained customers?

### Q5 — Compare daily viewing between stayed and left customers
Use a t-test to check whether the difference in average viewing is statistically significant.


In [59]:
from scipy.stats import ttest_ind

# Select retained customers' daily watch time
churned_stayed = dataset[dataset["churned"] == 0]["avg_watch_recalculated"]

# Select churned customers' daily watch time
churned_left = dataset[dataset["churned"] == 1]["avg_watch_recalculated"]

# Compare daily watch time between the two groups
ttest_ind(churned_stayed, churned_left)

TtestResult(statistic=np.float64(23.83460702786718), pvalue=np.float64(5.122041583121892e-119), df=np.float64(4998.0))

# Comparing warning signals
# How does churn compare among low-viewing customers, long-inactive customers,and customers showing both signals?

### Q6 — Compare possible churn warning signs
Check low viewing, long inactivity and both together to see which group has more churn.


In [70]:
# 1. Customers inactive for more than 30 days
last_login_days = dataset[
    dataset["last_login_days"] > 30
]

last_login_days_left = last_login_days[
    last_login_days["churn_status"] == "Left"
]

last_login_days_count = len(last_login_days_left)

left_login_rate = (
    last_login_days_count / len(last_login_days)
) * 100


# 2. Low-viewing customers
viewing_level = dataset[
    dataset["viewing_level"] == "Low"
]

viewing_level_left = viewing_level[
    viewing_level["churn_status"] == "Left"
]

viewing_level_count = len(viewing_level_left)

view_rate = (
    viewing_level_count / len(viewing_level)
) * 100


# 3. Customers showing BOTH warning signals
both_signals = dataset[
    (dataset["last_login_days"] > 30) &
    (dataset["viewing_level"] == "Low")
]

both_signals_left = both_signals[
    both_signals["churn_status"] == "Left"
]

both_signals_count = len(both_signals_left)

both_churn_rate = (
    both_signals_count / len(both_signals)
) * 100


# 4. Compare all three groups
print("Long-inactive customers who churned:", last_login_days_count)
print("Long-inactive churn rate:", round(left_login_rate, 2), "%")

print("Low-viewing customers who churned:", viewing_level_count)
print("Low-viewing churn rate:", round(view_rate, 2), "%")

print("Both-signals customers who churned:", both_signals_count)
print("Both-signals churn rate:", round(both_churn_rate, 2), "%")

Long-inactive customers who churned: 1774
Long-inactive churn rate: 75.39 %
Low-viewing customers who churned: 1068
Low-viewing churn rate: 85.58 %
Both-signals customers who churned: 583
Both-signals churn rate: 100.0 %


# Plan type
# Is the Basic plan inherently higher risk, or does it simply contain less-engaged customers? Churn is compared across plans within similar viewing levels

### Q7 — Count low-viewing customers who left by plan
Keep viewing level the same so we can compare Basic, Standard and Premium more fairly.


In [82]:
Basic_left=dataset.loc[(dataset["subscription_type"]=="Basic")&(dataset["viewing_level"]=="Low")&(dataset["churn_status"]=="Left")]
Standard_left=dataset.loc[(dataset["subscription_type"]=="Standard")&(dataset["viewing_level"]=="Low")&(dataset["churn_status"]=="Left")]
Premium_left=dataset.loc[(dataset["subscription_type"]=="Premium")&(dataset["viewing_level"]=="Low")&(dataset["churn_status"]=="Left")]


In [83]:
Basic_left_count=len(Basic_left)
Standard_left_count=len(Standard_left)
Premium_left_count=len(Premium_left)

print("The count of Basic Plan is: ",Basic_left_count)
print("The count of Standard Plan is: ",Standard_left_count)
print("The count of Premium Plan is: ",Premium_left_count)

The count of Basic Plan is:  404
The count of Standard Plan is:  337
The count of Premium Plan is:  327


### Count low-viewing customers who stayed by plan
We need the stayed counts as well to work out churn rates.


In [84]:
Basic_stayed=dataset.loc[(dataset["subscription_type"]=="Basic")&(dataset["viewing_level"]=="Low")&(dataset["churn_status"]=="Stayed")]
Standard_stayed=dataset.loc[(dataset["subscription_type"]=="Standard")&(dataset["viewing_level"]=="Low")&(dataset["churn_status"]=="Stayed")]
Premium_stayed=dataset.loc[(dataset["subscription_type"]=="Premium")&(dataset["viewing_level"]=="Low")&(dataset["churn_status"]=="Stayed")]

In [85]:
Basic_stayed_count=len(Basic_stayed)
Standard_stayed_count=len(Standard_stayed)
Premium_stayed_count=len(Premium_stayed)

print("The count of Basic Plan is: ",Basic_stayed_count)
print("The count of Standard Plan is: ",Standard_stayed_count)
print("The count of Premium Plan is: ",Premium_stayed_count)

The count of Basic Plan is:  10
The count of Standard Plan is:  86
The count of Premium Plan is:  84


### Find the total customers in each plan
Add left and stayed customers to get the correct denominator.


In [88]:
total_basic=Basic_left_count+Basic_stayed_count
total_standard=Standard_left_count+Standard_stayed_count
total_premium=Premium_left_count+Premium_stayed_count

In [87]:
total_basic

414

### Calculate plan churn rates within low viewing
Compare the percentage who left, not just the number who left.


In [89]:
churn_rate_basic = (Basic_left_count / total_basic) * 100
churn_rate_standard = (Standard_left_count / total_standard) * 100
churn_rate_premium = (Premium_left_count / total_premium) * 100

print("The Churn Rate for Basic Plan is:", round(churn_rate_basic, 2), "%")
print("The Churn Rate for Standard Plan is:", round(churn_rate_standard, 2), "%")
print("The Churn Rate for Premium Plan is:", round(churn_rate_premium, 2), "%")

The Churn Rate for Basic Plan is: 97.58 %
The Churn Rate for Standard Plan is: 79.67 %
The Churn Rate for Premium Plan is: 79.56 %


# Payment method
# Within similar viewing levels, do manual payers churn more than automatic payers?

### Q8 — Compare payment methods among low viewers
Count stayed and left customers for manual and automatic payments.


In [90]:
# Manual and Automatic payment customers who LEFT (Low viewing)

Manual_left = dataset.loc[
    (dataset["payment_type"] == "Manual") &
    (dataset["viewing_level"] == "Low") &
    (dataset["churn_status"] == "Left")
]

Automatic_left = dataset.loc[
    (dataset["payment_type"] == "Automatic") &
    (dataset["viewing_level"] == "Low") &
    (dataset["churn_status"] == "Left")
]

Manual_left_count = len(Manual_left)
Automatic_left_count = len(Automatic_left)

print("Manual payment customers who Left:", Manual_left_count)
print("Automatic payment customers who Left:", Automatic_left_count)


# Manual and Automatic payment customers who STAYED (Low viewing)

Manual_Stayed = dataset.loc[
    (dataset["payment_type"] == "Manual") &
    (dataset["viewing_level"] == "Low") &
    (dataset["churn_status"] == "Stayed")
]

Automatic_Stayed = dataset.loc[
    (dataset["payment_type"] == "Automatic") &
    (dataset["viewing_level"] == "Low") &
    (dataset["churn_status"] == "Stayed")
]

Manual_Stayed_count = len(Manual_Stayed)
Automatic_Stayed_count = len(Automatic_Stayed)

print("Manual payment customers who Stayed:", Manual_Stayed_count)
print("Automatic payment customers who Stayed:", Automatic_Stayed_count)


# Total customers in each payment group

total_Manual = Manual_left_count + Manual_Stayed_count
total_Automatic = Automatic_left_count + Automatic_Stayed_count


# Calculate churn rates

churn_rate_Manual = (Manual_left_count / total_Manual) * 100
churn_rate_Automatic = (Automatic_left_count / total_Automatic) * 100


# Display churn rates

print("The Churn Rate for Manual Payment:", round(churn_rate_Manual, 2), "%")
print("The Churn Rate for Automatic Payment:", round(churn_rate_Automatic, 2), "%")

Manual payment customers who Left: 445
Automatic payment customers who Left: 623
Manual payment customers who Stayed: 24
Automatic payment customers who Stayed: 156
The Churn Rate for Manual Payment: 94.88 %
The Churn Rate for Automatic Payment: 79.97 %


### Repeat the payment comparison for medium viewers
Check whether the same pattern appears at a different viewing level.


In [91]:
# MEDIUM VIEWING - LEFT

Manual_left_medium = dataset.loc[
    (dataset["payment_type"] == "Manual") &
    (dataset["viewing_level"] == "Medium") &
    (dataset["churn_status"] == "Left")
]

Automatic_left_medium = dataset.loc[
    (dataset["payment_type"] == "Automatic") &
    (dataset["viewing_level"] == "Medium") &
    (dataset["churn_status"] == "Left")
]

Manual_left_medium_count = len(Manual_left_medium)
Automatic_left_medium_count = len(Automatic_left_medium)

# MEDIUM VIEWING - STAYED

Manual_stayed_medium = dataset.loc[
    (dataset["payment_type"] == "Manual") &
    (dataset["viewing_level"] == "Medium") &
    (dataset["churn_status"] == "Stayed")
]

Automatic_stayed_medium = dataset.loc[
    (dataset["payment_type"] == "Automatic") &
    (dataset["viewing_level"] == "Medium") &
    (dataset["churn_status"] == "Stayed")
]

Manual_stayed_medium_count = len(Manual_stayed_medium)
Automatic_stayed_medium_count = len(Automatic_stayed_medium)

# TOTAL CUSTOMERS

total_Manual_medium = Manual_left_medium_count + Manual_stayed_medium_count
total_Automatic_medium = Automatic_left_medium_count + Automatic_stayed_medium_count

# CHURN RATES

churn_rate_Manual_medium = (Manual_left_medium_count / total_Manual_medium) * 100
churn_rate_Automatic_medium = (Automatic_left_medium_count / total_Automatic_medium) * 100

# RESULTS

print("MEDIUM VIEWING LEVEL")
print("Manual payment customers who Left:", Manual_left_medium_count)
print("Manual payment customers who Stayed:", Manual_stayed_medium_count)
print("Total Manual customers:", total_Manual_medium)
print("Manual churn rate:", round(churn_rate_Manual_medium, 2), "%")

print("Automatic payment customers who Left:", Automatic_left_medium_count)
print("Automatic payment customers who Stayed:", Automatic_stayed_medium_count)
print("Total Automatic customers:", total_Automatic_medium)
print("Automatic churn rate:", round(churn_rate_Automatic_medium, 2), "%")

MEDIUM VIEWING LEVEL
Manual payment customers who Left: 575
Manual payment customers who Stayed: 376
Total Manual customers: 951
Manual churn rate: 60.46 %
Automatic payment customers who Left: 687
Automatic payment customers who Stayed: 864
Total Automatic customers: 1551
Automatic churn rate: 44.29 %


### Repeat the payment comparison for high viewers
See whether manual payment still has higher churn among engaged viewers.


In [92]:
# HIGH VIEWING - LEFT

Manual_left_high = dataset.loc[
    (dataset["payment_type"] == "Manual") &
    (dataset["viewing_level"] == "High") &
    (dataset["churn_status"] == "Left")
]

Automatic_left_high = dataset.loc[
    (dataset["payment_type"] == "Automatic") &
    (dataset["viewing_level"] == "High") &
    (dataset["churn_status"] == "Left")
]

Manual_left_high_count = len(Manual_left_high)
Automatic_left_high_count = len(Automatic_left_high)

# HIGH VIEWING - STAYED

Manual_stayed_high = dataset.loc[
    (dataset["payment_type"] == "Manual") &
    (dataset["viewing_level"] == "High") &
    (dataset["churn_status"] == "Stayed")
]

Automatic_stayed_high = dataset.loc[
    (dataset["payment_type"] == "Automatic") &
    (dataset["viewing_level"] == "High") &
    (dataset["churn_status"] == "Stayed")
]

Manual_stayed_high_count = len(Manual_stayed_high)
Automatic_stayed_high_count = len(Automatic_stayed_high)

# TOTAL CUSTOMERS

total_Manual_high = Manual_left_high_count + Manual_stayed_high_count
total_Automatic_high = Automatic_left_high_count + Automatic_stayed_high_count

# CHURN RATES

churn_rate_Manual_high = (Manual_left_high_count / total_Manual_high) * 100
churn_rate_Automatic_high = (Automatic_left_high_count / total_Automatic_high) * 100

# RESULTS

print("HIGH VIEWING LEVEL")
print("Manual payment customers who Left:", Manual_left_high_count)
print("Manual payment customers who Stayed:", Manual_stayed_high_count)
print("Total Manual customers:", total_Manual_high)
print("Manual churn rate:", round(churn_rate_Manual_high, 2), "%")

print("Automatic payment customers who Left:", Automatic_left_high_count)
print("Automatic payment customers who Stayed:", Automatic_stayed_high_count)
print("Total Automatic customers:", total_Automatic_high)
print("Automatic churn rate:", round(churn_rate_Automatic_high, 2), "%")

HIGH VIEWING LEVEL
Manual payment customers who Left: 96
Manual payment customers who Stayed: 389
Total Manual customers: 485
Manual churn rate: 19.79 %
Automatic payment customers who Left: 89
Automatic payment customers who Stayed: 676
Total Automatic customers: 765
Automatic churn rate: 11.63 %


# Account sharing
# How does churn vary with the number of profiles, and at what point do additional profiles stop improving retention?

### Q9 — Check the available profile counts
See which numbers of profiles appear in the dataset.


In [97]:
Unique_profile=dataset["number_of_profiles"].unique()
print("The Unique Profiles: ",Unique_profile)

The Unique Profiles:  [1 5 2 3 4]


### Compare churn for customers with one profile
Calculate left, stayed and total so we can compare percentages.


In [98]:
# Customers with 1 profile who LEFT
profile_1_left = dataset.loc[
    (dataset["number_of_profiles"] == 1) &
    (dataset["churn_status"] == "Left")
]

profile_1_left_count = len(profile_1_left)


# Customers with 1 profile who STAYED
profile_1_Stayed = dataset.loc[
    (dataset["number_of_profiles"] == 1) &
    (dataset["churn_status"] == "Stayed")
]

profile_1_Stayed_count = len(profile_1_Stayed)


# Total customers with 1 profile
total_profile_1 = profile_1_left_count + profile_1_Stayed_count


# Calculate churn and retention rates
churn_rate_profile_1_left = (profile_1_left_count / total_profile_1) * 100

retention_rate_profile_1 = (profile_1_Stayed_count / total_profile_1) * 100


# Display results
print("Customers with 1 profile who Left:", profile_1_left_count)
print("Customers with 1 profile who Stayed:", profile_1_Stayed_count)
print("Total customers with 1 profile:", total_profile_1)

print("Churn rate for 1 profile:", round(churn_rate_profile_1_left, 2), "%")
print("Retention rate for 1 profile:", round(retention_rate_profile_1, 2), "%")

Customers with 1 profile who Left: 570
Customers with 1 profile who Stayed: 402
Total customers with 1 profile: 972
Churn rate for 1 profile: 58.64 %
Retention rate for 1 profile: 41.36 %


### Compare churn for customers with two profiles
Use the same calculation for a fair comparison.


In [99]:
# Customers with 2 profiles who LEFT
profile_2_left = dataset.loc[
    (dataset["number_of_profiles"] == 2) &
    (dataset["churn_status"] == "Left")
]

profile_2_left_count = len(profile_2_left)


# Customers with 2 profiles who STAYED
profile_2_Stayed = dataset.loc[
    (dataset["number_of_profiles"] == 2) &
    (dataset["churn_status"] == "Stayed")
]

profile_2_Stayed_count = len(profile_2_Stayed)


# Total customers with 2 profiles
total_profile_2 = profile_2_left_count + profile_2_Stayed_count


# Calculate churn and retention rates
churn_rate_profile_2_left = (profile_2_left_count / total_profile_2) * 100

retention_rate_profile_2 = (profile_2_Stayed_count / total_profile_2) * 100


# Display results
print("Customers with 2 profiles who Left:", profile_2_left_count)
print("Customers with 2 profiles who Stayed:", profile_2_Stayed_count)
print("Total customers with 2 profiles:", total_profile_2)

print("Churn rate for 2 profiles:", round(churn_rate_profile_2_left, 2), "%")
print("Retention rate for 2 profiles:", round(retention_rate_profile_2, 2), "%")

Customers with 2 profiles who Left: 575
Customers with 2 profiles who Stayed: 426
Total customers with 2 profiles: 1001
Churn rate for 2 profiles: 57.44 %
Retention rate for 2 profiles: 42.56 %


### Compare churn for customers with three profiles
Use the same calculation for a fair comparison.


In [100]:
# Customers with 3 profiles who LEFT
profile_3_left = dataset.loc[
    (dataset["number_of_profiles"] == 3) &
    (dataset["churn_status"] == "Left")
]

profile_3_left_count = len(profile_3_left)

# Customers with 3 profiles who STAYED
profile_3_Stayed = dataset.loc[
    (dataset["number_of_profiles"] == 3) &
    (dataset["churn_status"] == "Stayed")
]

profile_3_Stayed_count = len(profile_3_Stayed)

# Total customers with 3 profiles
total_profile_3 = profile_3_left_count + profile_3_Stayed_count

# Calculate churn and retention rates
churn_rate_profile_3_left = (profile_3_left_count / total_profile_3) * 100
retention_rate_profile_3 = (profile_3_Stayed_count / total_profile_3) * 100

# Display results
print("Customers with 3 profiles who Left:", profile_3_left_count)
print("Customers with 3 profiles who Stayed:", profile_3_Stayed_count)
print("Total customers with 3 profiles:", total_profile_3)
print("Churn rate for 3 profiles:", round(churn_rate_profile_3_left, 2), "%")
print("Retention rate for 3 profiles:", round(retention_rate_profile_3, 2), "%")

Customers with 3 profiles who Left: 575
Customers with 3 profiles who Stayed: 419
Total customers with 3 profiles: 994
Churn rate for 3 profiles: 57.85 %
Retention rate for 3 profiles: 42.15 %


### Compare churn for customers with four profiles
Use the same calculation for a fair comparison.


In [101]:
# Customers with 4 profiles who LEFT
profile_4_left = dataset.loc[
    (dataset["number_of_profiles"] == 4) &
    (dataset["churn_status"] == "Left")
]

profile_4_left_count = len(profile_4_left)

# Customers with 4 profiles who STAYED
profile_4_Stayed = dataset.loc[
    (dataset["number_of_profiles"] == 4) &
    (dataset["churn_status"] == "Stayed")
]

profile_4_Stayed_count = len(profile_4_Stayed)

# Total customers with 4 profiles
total_profile_4 = profile_4_left_count + profile_4_Stayed_count

# Calculate churn and retention rates
churn_rate_profile_4_left = (profile_4_left_count / total_profile_4) * 100
retention_rate_profile_4 = (profile_4_Stayed_count / total_profile_4) * 100

# Display results
print("Customers with 4 profiles who Left:", profile_4_left_count)
print("Customers with 4 profiles who Stayed:", profile_4_Stayed_count)
print("Total customers with 4 profiles:", total_profile_4)
print("Churn rate for 4 profiles:", round(churn_rate_profile_4_left, 2), "%")
print("Retention rate for 4 profiles:", round(retention_rate_profile_4, 2), "%")

Customers with 4 profiles who Left: 375
Customers with 4 profiles who Stayed: 624
Total customers with 4 profiles: 999
Churn rate for 4 profiles: 37.54 %
Retention rate for 4 profiles: 62.46 %


### Compare churn for customers with five profiles
Use the same calculation for a fair comparison.


In [102]:
# Customers with 5 profiles who LEFT
profile_5_left = dataset.loc[
    (dataset["number_of_profiles"] == 5) &
    (dataset["churn_status"] == "Left")
]

profile_5_left_count = len(profile_5_left)

# Customers with 5 profiles who STAYED
profile_5_Stayed = dataset.loc[
    (dataset["number_of_profiles"] == 5) &
    (dataset["churn_status"] == "Stayed")
]

profile_5_Stayed_count = len(profile_5_Stayed)

# Total customers with 5 profiles
total_profile_5 = profile_5_left_count + profile_5_Stayed_count

# Calculate churn and retention rates
churn_rate_profile_5_left = (profile_5_left_count / total_profile_5) * 100
retention_rate_profile_5 = (profile_5_Stayed_count / total_profile_5) * 100

# Display results
print("Customers with 5 profiles who Left:", profile_5_left_count)
print("Customers with 5 profiles who Stayed:", profile_5_Stayed_count)
print("Total customers with 5 profiles:", total_profile_5)
print("Churn rate for 5 profiles:", round(churn_rate_profile_5_left, 2), "%")
print("Retention rate for 5 profiles:", round(retention_rate_profile_5, 2), "%")

Customers with 5 profiles who Left: 420
Customers with 5 profiles who Stayed: 614
Total customers with 5 profiles: 1034
Churn rate for 5 profiles: 40.62 %
Retention rate for 5 profiles: 59.38 %


# Non-influential factors
# Are gender, region and device associated with churn?

### Q9B — Check the categories before testing
Look at the gender, region and device groups in the data.


In [104]:
gender_unique=dataset["gender"].unique()
region_unique=dataset["region"].unique()
device_unique=dataset["device"].unique()
print("Gender",gender_unique)
print("Region",region_unique)
print("Device",device_unique)

Gender ['Other' 'Female' 'Male']
Region ['Africa' 'Europe' 'Asia' 'Oceania' 'South America' 'North America']
Device ['TV' 'Mobile' 'Laptop' 'Tablet' 'Desktop']


### Count churn for female customers
Get the stayed and left counts before comparing genders.


In [105]:
# ==================================================
# QUESTION 9B: GENDER VS CUSTOMER CHURN
# CATEGORY: FEMALE
# ==================================================

# Female customers who LEFT
Female_left = dataset.loc[
    (dataset["gender"] == "Female") &
    (dataset["churn_status"] == "Left")
]

Female_left_count = len(Female_left)


# Female customers who STAYED
Female_stayed = dataset.loc[
    (dataset["gender"] == "Female") &
    (dataset["churn_status"] == "Stayed")
]

Female_stayed_count = len(Female_stayed)


# Total Female customers
total_Female = Female_left_count + Female_stayed_count


# Calculate churn and retention rates
churn_rate_Female = (Female_left_count / total_Female) * 100
retention_rate_Female = (Female_stayed_count / total_Female) * 100


# Display results
print("QUESTION 9B: GENDER VS CUSTOMER CHURN — FEMALE")
print("Female customers who Left:", Female_left_count)
print("Female customers who Stayed:", Female_stayed_count)
print("Total Female customers:", total_Female)
print("Female churn rate:", round(churn_rate_Female, 2), "%")
print("Female retention rate:", round(retention_rate_Female, 2), "%")

QUESTION 9B: GENDER VS CUSTOMER CHURN — FEMALE
Female customers who Left: 874
Female customers who Stayed: 837
Total Female customers: 1711
Female churn rate: 51.08 %
Female retention rate: 48.92 %


### Count churn for male customers
Use the same approach for the next gender group.


In [106]:
# ==================================================
# QUESTION 9B: GENDER VS CUSTOMER CHURN
# CATEGORY: MALE
# ==================================================

# Male customers who LEFT
Male_left = dataset.loc[
    (dataset["gender"] == "Male") &
    (dataset["churn_status"] == "Left")
]

Male_left_count = len(Male_left)


# Male customers who STAYED
Male_stayed = dataset.loc[
    (dataset["gender"] == "Male") &
    (dataset["churn_status"] == "Stayed")
]

Male_stayed_count = len(Male_stayed)


# Total Male customers
total_Male = Male_left_count + Male_stayed_count


# Calculate churn and retention rates
churn_rate_Male = (Male_left_count / total_Male) * 100
retention_rate_Male = (Male_stayed_count / total_Male) * 100


# Display results
print("QUESTION 9B: GENDER VS CUSTOMER CHURN — MALE")
print("Male customers who Left:", Male_left_count)
print("Male customers who Stayed:", Male_stayed_count)
print("Total Male customers:", total_Male)
print("Male churn rate:", round(churn_rate_Male, 2), "%")
print("Male retention rate:", round(retention_rate_Male, 2), "%")

QUESTION 9B: GENDER VS CUSTOMER CHURN — MALE
Male customers who Left: 827
Male customers who Stayed: 827
Total Male customers: 1654
Male churn rate: 50.0 %
Male retention rate: 50.0 %


### Count churn for customers in the Other category
Finish the manual counts for the gender comparison.


In [107]:
# ==================================================
# QUESTION 9B: GENDER VS CUSTOMER CHURN
# CATEGORY: OTHER
# ==================================================

# Other gender customers who LEFT
Other_left = dataset.loc[
    (dataset["gender"] == "Other") &
    (dataset["churn_status"] == "Left")
]

Other_left_count = len(Other_left)


# Other gender customers who STAYED
Other_stayed = dataset.loc[
    (dataset["gender"] == "Other") &
    (dataset["churn_status"] == "Stayed")
]

Other_stayed_count = len(Other_stayed)


# Total Other gender customers
total_Other = Other_left_count + Other_stayed_count


# Calculate churn and retention rates
churn_rate_Other = (Other_left_count / total_Other) * 100
retention_rate_Other = (Other_stayed_count / total_Other) * 100


# Display results
print("QUESTION 9B: GENDER VS CUSTOMER CHURN — OTHER")
print("Other customers who Left:", Other_left_count)
print("Other customers who Stayed:", Other_stayed_count)
print("Total Other customers:", total_Other)
print("Other churn rate:", round(churn_rate_Other, 2), "%")
print("Other retention rate:", round(retention_rate_Other, 2), "%")

QUESTION 9B: GENDER VS CUSTOMER CHURN — OTHER
Other customers who Left: 814
Other customers who Stayed: 821
Total Other customers: 1635
Other churn rate: 49.79 %
Other retention rate: 50.21 %


### Make the gender churn table
Put all gender groups into one table so we can test for an association.


In [108]:
gender_table = pd.crosstab(dataset["gender"], dataset["churn_status"])

In [109]:
gender_table

churn_status,Left,Stayed
gender,,
Female,874,837
Male,827,827
Other,814,821


### Run the chi-square test for gender
Use the p-value to check whether the observed differences are statistically significant.


In [112]:
from scipy.stats import chi2_contingency
chi2, p_value, dof, expected = chi2_contingency(gender_table)
print(expected,p_value)

[[860.633 850.367]
 [831.962 822.038]
 [822.405 812.595]] 0.7224877191844741


### Show the expected counts
These are the counts we would expect if gender and churn were independent.


In [111]:
expected_table = pd.DataFrame(
    expected,
    index=gender_table.index,
    columns=gender_table.columns
)

print(expected_table.round(2))

churn_status    Left  Stayed
gender                      
Female        860.63  850.37
Male          831.96  822.04
Other         822.40  812.60


### Make the region churn table
Count stayed and left customers in each region.


In [113]:
region_table = pd.crosstab(dataset["region"], dataset["churn_status"])

In [114]:
region_table

churn_status,Left,Stayed
region,,
Africa,373,403
Asia,413,405
Europe,430,405
North America,406,420
Oceania,369,372
South America,524,480


### Run the chi-square test for region
Check whether region and churn show a statistically significant association.


In [115]:
chi2, p_value, dof, expected = chi2_contingency(region_table)

print("Chi-square:", chi2)
print("P-value:", p_value)

Chi-square: 3.9844005928389765
P-value: 0.5516640727675917


### Make the device churn table
Count stayed and left customers for each device.


In [116]:
device_table = pd.crosstab(dataset["device"], dataset["churn_status"])

In [117]:
device_table

churn_status,Left,Stayed
device,,
Desktop,447,462
Laptop,494,466
Mobile,487,483
TV,473,480
Tablet,614,594


### Run the chi-square test for device
Check whether device and churn show a statistically significant association.


In [118]:
chi2, p_value, dof, expected = chi2_contingency(device_table)

print("Chi-square:", chi2)
print("P-value:", p_value)

Chi-square: 1.2832748694361091
P-value: 0.8642067410460219


# Atypical customers
# What characteristics distinguish heavy viewers who churned and light viewers who stayed?


### Q10 — Count unusual customer segments
See how many heavy viewers left and how many light viewers stayed.


In [119]:
segment_table = pd.crosstab(
    dataset["customer_segment"],
    dataset["churn_status"]
)

In [120]:
segment_table

churn_status,Left,Stayed
customer_segment,,
Heavy viewer who left,185,0
Light viewer who stayed,0,180
Typical,2330,2305


# Revenue loss
# Which plan or segment accounts for the greatest revenue lost to churn, and is it the same segment with the highest churn rate?

### Q11 — Calculate monthly revenue lost by plan
Add the monthly fees of churned customers in each plan to see which plan lost the most.


In [121]:
# Q11: Monthly revenue lost due to churn by subscription plan

# BASIC
Basic_left = dataset.loc[
    (dataset["subscription_type"] == "Basic") &
    (dataset["churn_status"] == "Left")
]["monthly_fee"]

basic_revenue_lost = Basic_left.sum()


# STANDARD
Standard_left = dataset.loc[
    (dataset["subscription_type"] == "Standard") &
    (dataset["churn_status"] == "Left")
]["monthly_fee"]

standard_revenue_lost = Standard_left.sum()


# PREMIUM
Premium_left = dataset.loc[
    (dataset["subscription_type"] == "Premium") &
    (dataset["churn_status"] == "Left")
]["monthly_fee"]

premium_revenue_lost = Premium_left.sum()


# DISPLAY RESULTS
print("Basic monthly revenue lost:", round(basic_revenue_lost, 2))
print("Standard monthly revenue lost:", round(standard_revenue_lost, 2))
print("Premium monthly revenue lost:", round(premium_revenue_lost, 2))

Basic monthly revenue lost: 9223.74
Standard monthly revenue lost: 10436.54
Premium monthly revenue lost: 13366.57


### Count stayed and left customers by plan
Revenue loss alone does not tell us which plan has the highest churn rate.


In [122]:
Sub_table=pd.crosstab(
    dataset["subscription_type"],
    dataset["churn_status"]
)

In [123]:
Sub_table

churn_status,Left,Stayed
subscription_type,,
Basic,1026,634
Premium,743,954
Standard,746,897


### Calculate churn rate for each plan
Divide the left count by the total for each plan so we can compare the rates.


In [124]:
Sub_table["Total"] = Sub_table["Left"] + Sub_table["Stayed"]

Sub_table["Churn_Rate"] = (
    Sub_table["Left"] / Sub_table["Total"] * 100
).round(2)

print(Sub_table)

churn_status       Left  Stayed  Total  Churn_Rate
subscription_type                                 
Basic              1026     634   1660       61.81
Premium             743     954   1697       43.78
Standard            746     897   1643       45.40


# Retention priority
# Ranking vulnerable segments by churn risk, segment size, revenue at risk and feasibility of intervention, which single customer group should be prioritised for retention?

### Q12 — Start with the low-viewing group
Count how many low-viewing customers stayed and left.


In [125]:
viewing_table = pd.crosstab(
    dataset["viewing_level"],
    dataset["churn_status"]
)

print(viewing_table)

churn_status   Left  Stayed
viewing_level              
High            185    1065
Low            1068     180
Medium         1262    1240


### Check monthly revenue still retained from low viewers
Sum the fees of low-viewing customers who stayed.


In [126]:
low_viewing_stayed = dataset.loc[
    (dataset["viewing_level"] == "Low") &
    (dataset["churn_status"] == "Stayed")
]["monthly_fee"]

low_viewing_revenue = low_viewing_stayed.sum()

print("Low-viewing retained customers:", len(low_viewing_stayed))
print("Monthly revenue retained:", round(low_viewing_revenue, 2))

Low-viewing retained customers: 180
Monthly revenue retained: 2804.2


### Check the group inactive for over 30 days
Count stayed and left customers to measure the size and churn of this group.


In [127]:
inactive_table = pd.crosstab(
    dataset["last_login_days"] > 30,
    dataset["churn_status"]
)

print(inactive_table)

churn_status     Left  Stayed
last_login_days              
False             741    1906
True             1774     579


### Check monthly revenue retained from inactive customers
These customers have not logged in recently but are still subscribed.


In [128]:
inactive_revenue = dataset.loc[
    (dataset["last_login_days"] > 30) &
    (dataset["churn_status"] == "Stayed"),
    "monthly_fee"
].sum()

print("Monthly revenue retained:", round(inactive_revenue, 2))

Monthly revenue retained: 8591.21


### Check the manual-payment group
Count stayed and left customers to compare this group with the others.


In [129]:
payment_table = pd.crosstab(
    dataset["payment_type"],
    dataset["churn_status"]
)

print(payment_table)

churn_status  Left  Stayed
payment_type              
Automatic     1399    1696
Manual        1116     789


### Check monthly revenue retained from manual payments
Sum the monthly fees of manual-payment customers who stayed.


In [130]:
manual_revenue = dataset.loc[
    (dataset["payment_type"] == "Manual") &
    (dataset["churn_status"] == "Stayed"),
    "monthly_fee"
].sum()

print("Monthly revenue retained:", round(manual_revenue, 2))

Monthly revenue retained: 11181.11


### Check monthly revenue retained from Basic subscribers
Complete the retained-revenue comparison for the fourth group.


In [132]:
basic_revenue = dataset.loc[
    (dataset["subscription_type"] == "Basic") &
    (dataset["churn_status"] == "Stayed"),
    "monthly_fee"
].sum()

print("Monthly revenue retained:", round(basic_revenue, 2))

Monthly revenue retained: 5699.66


### Calculate monthly revenue lost across all four groups
Use the fees of customers who left so we can compare the historical revenue loss.


In [133]:

# Q12: Monthly revenue lost from churned customers

# 1. Low-viewing customers
low_left_revenue = dataset.loc[
    (dataset["viewing_level"] == "Low") &
    (dataset["churn_status"] == "Left"),
    "monthly_fee"
].sum()

# 2. Customers inactive for more than 30 days
inactive_left_revenue = dataset.loc[
    (dataset["last_login_days"] > 30) &
    (dataset["churn_status"] == "Left"),
    "monthly_fee"
].sum()

# 3. Manual-payment customers
manual_left_revenue = dataset.loc[
    (dataset["payment_type"] == "Manual") &
    (dataset["churn_status"] == "Left"),
    "monthly_fee"
].sum()

# 4. Basic subscribers
basic_left_revenue = dataset.loc[
    (dataset["subscription_type"] == "Basic") &
    (dataset["churn_status"] == "Left"),
    "monthly_fee"
].sum()

# Display results
print("Low viewing:", round(low_left_revenue, 2))
print("Inactive >30 days:", round(inactive_left_revenue, 2))
print("Manual payment:", round(manual_left_revenue, 2))
print("Basic subscribers:", round(basic_left_revenue, 2))


Low viewing: 14229.32
Inactive >30 days: 23515.26
Manual payment: 15223.84
Basic subscribers: 9223.74


### Bring the four groups into one comparison table
We compare group size, churn rate, retained revenue and lost revenue rather than choosing by churn rate alone. Inactive customers over 30 days stand out because this is the largest group and it has the highest historical monthly revenue loss of the four. This is why we chose it as our retention priority.


In [136]:
retention_table = pd.DataFrame({
    "Customer Group": [
        "Low viewing",
        "Inactive >30 days",
        "Manual payment",
        "Basic subscribers"
    ],
    "Group Size": [1248, 2353, 1905, 1660],
    "Churn Rate (%)": [85.58, 75.39, 58.58, 61.81],
    "Revenue Retained": [
        low_viewing_revenue,
        inactive_revenue,
        manual_revenue,
        basic_revenue
    ],
    "Revenue Lost": [
        low_left_revenue,
        inactive_left_revenue,
        manual_left_revenue,
        basic_left_revenue
    ]
})

display(retention_table.round(2))

,Customer Group,Group Size,Churn Rate (%),Revenue Retained,Revenue Lost
0,Low viewing,1248,85.58,2804.20,14229.32
1,Inactive >30 days,2353,75.39,8591.21,23515.26
2,Manual payment,1905,58.58,11181.11,15223.84
3,Basic subscribers,1660,61.81,5699.66,9223.74
